# Committor from an infretis StapleTIS Run (CG permeation, many CVs)

Fits the committor with NPNEq (`CommittorNE`, CuPy) to the paths of an **infretis StapleTIS** simulation and compares it with the exact committor.

**How the paths become training data** (the same code as `load_tis_data_cupy.py`: `tis_data.prepare_data`, which switches to `infretis_data.py` for infretis runs; details in `docs/npneq_tis_notes.md`, sections 3, 9 and 15):
- **Read with infretis' own definitions:** interfaces and swap settings from `infretis.toml`, one row per finished path from `infretis_data.txt`, the order parameters from `load/<pnr>/order.txt` (or an h5 archive), and each path's core in an ensemble from infretis' `StaplePath.get_sh_region`.
- **Cut at M:** an ensemble only keeps paths that cross its middle interface M, so the frames before that crossing are conditioned on their future. Every (path, ensemble) pair gives one piece: the path from its first frame past M (searched from the core start) to its end. Optionally also the *mirrored piece*: the part before the last M crossing, read backwards (`REVERSED`).
- **Weights:** infretis records per (path, ensemble) the occupancy `path_f` and the weight `path_w` it put into the swap matrix (0.5 for LML/RMR paths in the inner staple ensembles). The weighting has to undo how the sampler used w: with swaps (infinite swap or staple swaps) that is `path_f / path_w` (`'staple'`), with shooting only `path_f` (`'rejections'`). By default the loader picks this from the run (`WEIGHTINGS = [None]`); several weightings can be fitted side by side by listing them.

**The run:** coarse-grained (Martini 3) permeation of a small solute (ALA bead) through a DPPC bilayer, GROMACS, StapleTIS with infinite swap (`CG_permeation/infstapletis`; the P2C6 system is the same with another solute). The order parameter is λ = 10 (z_ligand − z_bilayer), in Å, from the bilayer centre of mass; A is the water on one side (λ ≤ the first interface) and B the water on the other (λ ≥ the last).

**Many CVs:** `order.txt` of the newer `OP.py` has a column per CV after the time: λ, the ligand's orientation θ, the number of waters around it, vz, the lateral position (x, y), the box length, and the PO4 shift, far shift, head-group and water-core counts. `CV_NAMES` names the columns and `CV_EXCLUDE` leaves out the ones that are no CVs: the lateral position (the bilayer is translation invariant), the box length (constant) and vz (odd under time reversal, and stored with the wrong sign on backward-integrated frames, so it is wrong for the mirrored pieces and for the fit). Constant columns are dropped automatically. **Edit `CV_NAMES` to match your `order.txt`** (the notebook checks the count). Every iteration of the fit draws one of the CVs at random, so the cost per iteration does not depend on how many CVs there are, but each one needs enough draws: scale `MAX_ITER` with the number of CVs (default 1000 per CV).

**No exact committor** (section 5 sets `q_exact_of = None`). Instead section 7 shows how q depends on every CV (mean q per bin of each CV, and how much of the variance of q each CV alone explains), which is the point of fitting with many CVs: the committor of the system is a function of the solvent and lipid response, not of λ alone.

## 1. Imports

In [ ]:
import argparse
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cupy as cp

plt.rc('font', size=10)
plt.rc('axes', titlesize=12)
plt.rc('axes', labelsize=12)
plt.rc('legend', fontsize=9)

# Run from examples/: the CuPy optimalrcs package and the path readers of the examples
REPO = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs
import tis_data
import infretis_data
import seed_runs
FIG = f'{REPO}/examples/figures_by_potential'

## 2. Settings

**Run and data**

| Name | Meaning |
|---|---|
| `TIS_DIR` | the infretis simulation folder (`infretis.toml`, `infretis_data.txt`, `load/`) |
| `H5_FILE` | order parameters from an `inft store_txt -order` archive instead of `load/` (`None`: use `load/`) |
| `LAMBDA_A`, `LAMBDA_B` | states A (λ ≤ λ_A) and B (λ ≥ λ_B); `None`: the first and last interface |
| `BURN_IN_FRACTION` | the first part of the rows of `infretis_data.txt` (the order in which the paths ended) is skipped; the initial load paths always are |
| `PATHS_PER_ENSEMBLE`, `THIN` | at most this many paths per ensemble, evenly spaced (`None`: all), after keeping every `THIN`-th |
| `CV_NAMES`, `CV_EXCLUDE` | names of the columns of `order.txt` (after the time, λ first) and the ones that are no CVs; the rest, minus constant columns, are the fit CVs |
| `ITER_PER_CV` | main-fit iterations per CV (`MAX_ITER` = this × the number of CVs) |
| `REVERSED` | also use the mirrored pieces (time-reversible dynamics at equilibrium) |
| `WEIGHTINGS` | the weightings to fit, one fit each: `None` = the run's default, `'staple'` = path_f / path_w, `'rejections'` = path_f, `'none'` = 1 per (path, ensemble). One entry is enough; list several to compare them |

**Fit** (the same for every weighting): `SEEDS` (which CV and envelope centre each iteration draws), a warm-up of `WARMUP_ITER` iterations with a wide envelope, then `MAX_ITER` iterations, in chunks of `CHUNK` after which the plots are shown and saved. `STOPPED_ENDS` (default `True`): a piece that ends outside A and B was stopped by the path's own end (a turn); the stopped Z_q then uses q at that end for windows that pass it (`set_stopped_traj_end`), and the A/B traps are made long enough that no window is dropped. `NY` is the degree of the polynomial basis, `LAZY_BASIS` evaluates it on the fly (GPU memory).

In [ ]:
TIS_DIR = f'{REPO}/examples/data/infretis_inputs/cg_perm'   # copy infretis.toml, infretis_data.txt and archive.h5 (or load/) of the run here
H5_FILE = f'{TIS_DIR}/archive.h5'  # None: read load/<pnr>/order.txt
LAMBDA_A, LAMBDA_B = None, None    # None: first and last interface
BURN_IN_FRACTION = 0.3
PATHS_PER_ENSEMBLE = 3000          # None: all paths after the burn-in
THIN = 1
REVERSED = True
WEIGHTINGS = [None]                # None: the run's default (path_f / path_w for infinite swap); e.g. ['staple', 'rejections'] to compare

# the columns of order.txt after the time (lambda first), in the order OP.py writes them; EDIT to match your order.txt
CV_NAMES = ['lambda', 'theta', 'n_water', 'vz', 'x', 'y', 'box_x', 'po4_shift', 'po4_shift_far', 'n_head', 'n_wcore']
CV_EXCLUDE = ['vz', 'x', 'y', 'box_x']     # no CVs: vz is odd under time reversal (and sign-flipped on backward frames), x and y translation invariant, box_x constant
CV_LABELS = {'lambda': '$\\lambda$ (z, $\\AA$)', 'theta': '$\\theta$ (deg)', 'n_water': 'waters within 0.55 nm',
             'po4_shift': 'PO4 shift', 'po4_shift_far': 'PO4 shift (far)', 'n_head': 'head-group beads', 'n_wcore': 'waters in the core'}

SEEDS = [0]
WARMUP_ITER, WARMUP_ENVELOPE = 2000, 1
ITER_PER_CV = 1000                 # the main fit runs ITER_PER_CV iterations for every CV (each iteration draws one CV)
CHUNK = 2500
ENVELOPE, GAMMA = 0.01, 0.1
STOPPED_ENDS = True                # stopped Z_q: a window passing a turn end of a piece uses q at that end
NY, LAZY_BASIS = 6, True           # basis degree; evaluate the basis on the fly (GPU memory)
PLOTS = ('metrics', 'tis')         # library plots after every chunk; also possible: 'feps', 'obs_pred', 'obs_pred_log'
LABEL = 'order parameter $\\lambda$ ($\\AA$)'
N_BINS = 80                        # bins along lambda for the profiles

RUN_NAME = os.path.basename(TIS_DIR.rstrip('/'))
OUT = f'{FIG}/cg_permeation/infretis_staple/{RUN_NAME}'
print('figures and results in', OUT)


## 3. The Run

What infretis wrote: the sampling settings, and per ensemble its window (L, M, R), the number of finished paths with a nonzero occupancy, their total occupancy `path_f`, and the share of that occupancy with `path_w` ≠ 1 (where `'staple'` and `'rejections'` differ).

**Weight check** (`infretis_data.weight_consistency`, needs no committor). A path that is valid in two ensembles e and e′ should split its occupancy in the same ratio C_e / C_e′ whatever its type. So the ratio Σ weight_e / Σ weight_e′ over paths of one class (w_e, w_e′) must not depend on the class: relative to class (1, 1), the classes (0.5, 1) and (1, 0.5) should give 1 for the right weighting. Runs with swaps show which one that is. Runs with shooting only have no paths shared between ensembles, and the check is empty.

In [ ]:
settings = infretis_data.read_settings(TIS_DIR)
table = infretis_data.read_infretis_data(os.path.join(TIS_DIR, 'infretis_data.txt'), len(settings['interfaces']))
default_w, why = infretis_data.default_weighting(settings)
print(f"mode {settings['mode']}, infinite swap {settings['infinite_swap']}, staple swap fraction "
      f"{settings['staple_swap_frac']}, {len(table['pnr'])} finished paths")
print(f"interfaces {settings['interfaces']}")
print(f"default weighting: {default_w} ({why})")

f, w = table['path_f'], table['path_w']
rows = []
for e in range(len(settings['interfaces'])):
    on = f[:, e] > 0
    rows.append(dict(ensemble=f'{e:03d}', window=tuple(round(v, 4) for v in infretis_data.ensemble_window(settings, e)),
                     paths=int(on.sum()), occupancy=f[on, e].sum(),
                     share_w_not_1=(f[on, e] * ~np.isclose(w[on, e], 1)).sum() / max(f[on, e].sum(), 1e-12)))
display(pd.DataFrame(rows).set_index('ensemble').round(3))

check = infretis_data.weight_consistency(table)
w_not_1 = ((f > 0) & ~np.isclose(w, 1)).any()
if not w_not_1:
    print("path_w is 1 for every occupied (path, ensemble) pair (no wire fencing, no staple weights): 'staple' (path_f / path_w) "
          "and 'rejections' (path_f) are the same weights for this run, and the weight check below has nothing to compare")
if not check:
    print('weight check: no class of paths with w != 1 that is shared by two ensembles' if not w_not_1 else
          'weight check: no pairs of ensembles with enough shared paths (expected for shooting only)')
else:
    out = []
    for row in check:
        for c in ((0.5, 1), (1, 0.5)):
            if c in row:
                out.append(dict(pair=row['pair'], cls=c, n=row['n'][c],
                                path_f=f"{row[c]['rejections'][0]:.3f} ± {row[c]['rejections'][1]:.3f}",
                                path_f_over_w=f"{row[c]['staple'][0]:.3f} ± {row[c]['staple'][1]:.3f}"))
    display(pd.DataFrame(out))

## 4. From Paths to Frames

`tis_data.prepare_data` reads the run once per weighting in `WEIGHTINGS` (the pieces are the same, only their weights differ). It returns per frame: the CVs, λ, the time within its path (`t_traj`), its piece (`i_traj`), its path (`group_traj`, the infretis path number), the path weight (`path_weights`, mean 1) and the A/B masks.

The plots show where the frames are: the weighted number of frames along λ (the path-sampling density, high at the interfaces where pieces start, not the equilibrium distribution), and the piece lengths.

In [ ]:
def select_cvs(data):
    """Names of the fit CVs: the columns of order.txt that are not excluded and not constant."""
    n_col = data.X_all.shape[1]
    if n_col != len(CV_NAMES):
        raise SystemExit(f'order.txt has {n_col} columns after the time but CV_NAMES has {len(CV_NAMES)}: edit CV_NAMES in the settings')
    fr = ~(data.boundary0 | data.boundary1)
    names = []
    for j, n in enumerate(CV_NAMES):
        if n in CV_EXCLUDE:
            continue
        if np.ptp(data.X_all[fr, j]) == 0:
            print(f'dropping the constant column {n}')
            continue
        names.append(n)
    return names


def prepare(weighting):
    parser = argparse.ArgumentParser()
    tis_data.add_data_arguments(parser)
    argv = [TIS_DIR, '--burn-in-fraction', str(BURN_IN_FRACTION), '--thin', str(THIN)]
    if weighting is not None:
        argv += ['--weights', weighting]
    if H5_FILE is not None:
        argv += ['--h5', H5_FILE]
    if LAMBDA_A is not None:
        argv += ['--lambda-a', str(LAMBDA_A)]
    if LAMBDA_B is not None:
        argv += ['--lambda-b', str(LAMBDA_B)]
    if PATHS_PER_ENSEMBLE is not None:
        argv += ['--paths-per-ensemble', str(PATHS_PER_ENSEMBLE)]
    if REVERSED:
        argv += ['--reversed']
    args = parser.parse_args(argv)
    data = tis_data.prepare_data(args)
    data.cv_names = select_cvs(data)
    data.X = data.X_all[:, [CV_NAMES.index(n) for n in data.cv_names]]    # the CVs of the fit (X_all keeps every column of order.txt)
    return args.weights, args.lambda_a, args.lambda_b, data     # the weighting actually used


# None = the run's default (section 3); the same weighting twice is fitted once
names = list(dict.fromkeys(w if w is not None else default_w for w in WEIGHTINGS))
datas = {}
for weighting in names:
    name, lambda_a, lambda_b, data = prepare(weighting)
    datas[name] = data
    print(f'--> {name}: {len(data.lam)} frames, {data.i_traj.max() + 1} pieces, '
          f'{len(np.unique(data.group_traj))} paths\n')
data = next(iter(datas.values()))  # the frames are the same for every weighting
print(f'fit CVs ({data.X.shape[1]}): ' + (', '.join(data.cv_names) if hasattr(data, 'cv_names') else f'{data.X.shape[1]} column(s) of order.txt'))
free = ~(data.boundary0 | data.boundary1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
e_lam = np.linspace(data.lam.min(), data.lam.max(), 121)
for name, d in datas.items():
    ax1.hist(d.lam, bins=e_lam, weights=d.path_weights, histtype='step', label=name)
for v in settings['interfaces']:
    ax1.axvline(v, color='0.6', lw=0.6)
ax1.set(xlabel=LABEL, ylabel='weighted frames', yscale='log')
ax1.legend(title='weighting')
lengths = np.bincount(data.i_traj)
ax2.hist(lengths, bins=np.logspace(0, np.log10(lengths.max() + 1), 60))
ax2.set(xscale='log', yscale='log', xlabel='piece length (frames)', ylabel='pieces')
fig.tight_layout()
plt.show()

## 5. Exact Committor

None: there is no reference for an explicit-solvent MD system (the exact committor of the toy systems comes from the engine's discrete chain, which is known there). `q_exact_of = None` makes the fit skip the RMS against the exact committor; the plots then show only the fit.


In [ ]:
q_exact_of = None


## 6. Fit the Committor

One fit per weighting (and per seed in `SEEDS`), on the same frames. `CommittorNE` with `group_traj` (the pieces of one path count as one unit in the error estimates), the **A/B ends as traps** (`set_fixed_traj_length_trap`: longer than any piece plus the longest Z_q lag, so no window is dropped), and the **stopped ends** (`set_stopped_traj_end`, `STOPPED_ENDS`): a piece that ends outside A and B was stopped by the path's own end (a turn), so a window that passes it uses q at that end instead of being dropped, which would condition on the path not stopping earlier.

The first cell defines the plots, the progress line and the piece ends; the second builds the committor, runs `fit_transform` for the warm-up (wide envelope) and then in chunks of `CHUNK` iterations (it continues from the current q), and `seed_runs.save_and_show` shows and saves after the warm-up and every chunk as `<OUT>/<weighting>/seed<s>/it<N>_<plot>.png`:
- `metrics`: `dr2`, `zqs_z` and `|dx|` over the iterations. **Convergence: `zqs_z`** (the worst |stopped Z_q / standard error| over q and lags) about 2 or less, with `|dx|` small. For pieces cut at M the stopped Z_q of the committor is 0 at every q and lag (notes §5, §8.1);
- `tis`: the stopped Z_q per lag, the short-lag Z_q and the stopped observed-vs-predicted;
- `feps`, `obs_pred` (if listed in `PLOTS`): the library's original plots, biased for path pieces; with `STOPPED_ENDS` the Z_q of `feps` is the stopped one;
- `q_profile`: the path-weighted mean q per λ bin next to the exact committor, and fit − exact;

The progress line gives the path-weighted RMS(fit − exact) over the frames with 0.05 < q_exact < 0.95. Each weighting's final q per frame is kept in `fits` for section 7.

In [ ]:
def edges_for(d):
    return (np.linspace(lambda_a, lambda_b, N_BINS + 1),)


def extras_for(d, qe):
    # q -> figure: the path-weighted mean q per lambda bin, against the exact committor
    edges = edges_for(d)
    fr = ~(d.boundary0 | d.boundary1)
    bm = seed_runs.BinnedMean((d.lam,), edges[:1], fr)
    w_mean = bm(d.path_weights)
    mid = (edges[0][1:] + edges[0][:-1]) / 2
    prof_ex = None if qe is None else bm(d.path_weights * qe) / w_mean

    def q_profile(q):
        P = bm(d.path_weights * cp.asnumpy(q.r_traj)) / w_mean
        fig, axes = plt.subplots(1, 1 if qe is None else 2, figsize=(6 if qe is None else 13, 4), squeeze=False)
        axes[0, 0].plot(mid, P, label='fit')
        if qe is not None:
            axes[0, 0].plot(mid, prof_ex, 'k--', label='exact')
            t = (prof_ex > 0.05) & (prof_ex < 0.95)
            axes[0, 1].plot(mid, P - prof_ex)
            axes[0, 1].axhline(0, color='k', lw=0.8)
            axes[0, 1].set(xlabel=XLABEL, ylabel='fit $-$ exact',
                           title=f'RMS per bin (0.05 < exact < 0.95) {np.sqrt(np.nanmean((P - prof_ex)[t] ** 2)):.4f}')
            axes[0, 1].grid(alpha=0.3)
        axes[0, 0].set(xlabel=XLABEL, ylabel='$q$ (mean per bin)')
        axes[0, 0].legend()
        axes[0, 0].grid(alpha=0.3)
        fig.tight_layout()
        return fig

    extras = {'q_profile': q_profile}
    return extras


def report_for(d, qe):
    # progress line: path-weighted RMS(fit - exact) over the frames with 0.05 < q_exact < 0.95
    if qe is None:
        return None
    trans = ~(d.boundary0 | d.boundary1) & (qe > 0.05) & (qe < 0.95)
    idx, qe_g, w_g = (cp.asarray(a) for a in (np.flatnonzero(trans), qe[trans], d.path_weights[trans]))

    def report(q):
        diff = q.r_traj[idx] - qe_g
        return f'| RMS(fit - exact) {float(cp.sqrt(cp.sum(w_g * diff * diff) / cp.sum(w_g))):.4f}'
    return report


XLABEL = LABEL
MAX_ITER = ITER_PER_CV * data.X.shape[1]            # the main fit: ITER_PER_CV iterations per CV
print(f'fit with {data.X.shape[1]} CVs: {WARMUP_ITER} warm-up + {MAX_ITER} main iterations')

# ---- piece ends: in A or B (trap), or stopped by the path's own end, a turn (stopped end)
# (the frames are the same for every weighting, so this is done once)
b_all = data.boundary0 | data.boundary1
piece_end = np.r_[np.flatnonzero(np.diff(data.i_traj)), len(data.i_traj) - 1]      # last frame of every piece
stop_end = np.zeros(len(b_all), bool)
stop_end[piece_end] = ~b_all[piece_end]
longest = int(np.bincount(data.i_traj - data.i_traj.min()).max())
TRAP_LEN = max(2 ** 16, longest + 2 ** 15 + 1)         # A/B trap: longer than any piece plus the longest Z_q lag (2^15)
print(f'{len(piece_end)} pieces: {stop_end.sum()} end at a turn (stopped), {len(piece_end) - stop_end.sum()} in A or B; '
      f'longest piece {longest} frames, A/B trap {TRAP_LEN}')

PRINT_STEP = min(500, CHUNK, WARMUP_ITER or CHUNK)    # the metrics are recorded every PRINT_STEP iterations (a plot needs a record)
METRICS = ['delta_r2', 'max_z_zq_stopped', 'delta_x']


def show_stage(q, folder, n_done, stage, extras, report):
    """Save and show the plots after a stage (<folder>/it<N>_<plot>.png), and print the progress."""
    seed_runs.save_and_show(q, folder, f'it{n_done}', PLOTS, dict(metrics=METRICS), extras)
    print(f'{n_done} iterations ({stage}) {"" if report is None else report(q)}', flush=True)


In [ ]:
fit_kw = dict(ny=NY, print_step=PRINT_STEP, lazy_basis=LAZY_BASIS, save_min_metric='max_z_zq_stopped',
              metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x', 'time_elapsed'))
fits = {}                                      # weighting -> final q per seed, exact committor, fit time, zqs_z history
for name, d in datas.items():                  # one fit per weighting: the same frames, other path weights
    qe = q_exact_of(d.X_all) if q_exact_of is not None else None
    X_cv = cp.asarray(d.X)                     # the CVs on the GPU, once
    extras, report = extras_for(d, qe), report_for(d, qe)

    def comp_y():
        """The CV of an iteration: one of the fit CVs, drawn at random."""
        return X_cv[:, np.random.randint(X_cv.shape[1])]

    t0 = time.time()
    results = {}
    for seed in SEEDS:
        print(f'===== fit, weighting {name}, seed {seed} =====', flush=True)
        np.random.seed(seed)                   # which CV and envelope centre each iteration draws
        cp.random.seed(seed)
        folder = f'{OUT}/{name}/seed{seed}'

        # the committor of the path pieces
        q = optimalrcs.CommittorNE(
            boundary0=d.boundary0,             # state A: q = 0
            boundary1=d.boundary1,             # state B: q = 1
            i_traj=d.i_traj,                   # piece of every frame
            t_traj=d.t_traj,                   # time of every frame within its path
            path_weights=d.path_weights,       # weight of the frame's path (mean 1)
            group_traj=d.group_traj,           # path of every frame: its pieces are one unit in the error estimates
        )
        q.set_fixed_traj_length_trap(b_all, TRAP_LEN if STOPPED_ENDS else 2 ** 16)   # A/B ends: no window is dropped
        if STOPPED_ENDS:
            q.set_stopped_traj_end(stop_end)   # turn ends: a window passing the end uses q there

        # warm-up with a wide envelope
        n_done = 0
        if WARMUP_ITER:
            optimalrcs.envelope_scale = WARMUP_ENVELOPE
            q.fit_transform(comp_y, gamma=GAMMA, max_iter=WARMUP_ITER, **fit_kw)
            n_done += WARMUP_ITER
            show_stage(q, folder, n_done, f'warm-up, envelope {WARMUP_ENVELOPE}', extras, report)

        # main fit, in chunks of CHUNK iterations (fit_transform continues from the current q)
        optimalrcs.envelope_scale = ENVELOPE
        while n_done < WARMUP_ITER + MAX_ITER:
            n_iter = min(CHUNK, WARMUP_ITER + MAX_ITER - n_done)
            q.fit_transform(comp_y, gamma=GAMMA, max_iter=n_iter, **fit_kw)
            n_done += n_iter
            show_stage(q, folder, n_done, f'main, envelope {ENVELOPE}', extras, report)

        results[seed] = cp.asnumpy(q.r_traj)
        np.save(f'{folder}/q_traj.npy', results[seed].astype(np.float32))
        zqs_z = np.array([float(v) for v in q.metrics_history['max_z_zq_stopped']])   # (of the last seed, as before)
        del q                                  # free the GPU for the next fit
        cp.get_default_memory_pool().free_all_blocks()
    fits[name] = dict(r=results, q_exact=qe, t_fit=time.time() - t0, zqs_z=zqs_z)
    del X_cv

## 7. Results

**Table:** per weighting and seed, the data size, the final and lowest `zqs_z` of the run (its own data; about 2 or less is consistent), and with the exact committor the path-weighted RMS of fit − exact over the frames with 0.05 < q_exact < 0.95 and over the bins in that range.

**Committor along λ:** the path-weighted mean q per λ bin of every fit, the exact committor averaged the same way (dashed), and fit − exact.

In [ ]:
rows, profiles = [], {}
for name, fit in fits.items():
    d = datas[name]
    fr = ~(d.boundary0 | d.boundary1)
    edges = edges_for(d)
    bm = seed_runs.BinnedMean((d.lam,), edges[:1], fr)
    w_mean = bm(d.path_weights)
    qe = fit['q_exact']
    for s, r in fit['r'].items():
        row = dict(weighting=name, seed=s, paths=len(np.unique(d.group_traj)), pieces=int(d.i_traj.max() + 1),
                   frames=len(r), zqs_z_final=fit['zqs_z'][-1], zqs_z_min=fit['zqs_z'][1:].min(),
                   fit_minutes=fit['t_fit'] / 60 / len(fit['r']))
        profiles[(name, s)] = bm(d.path_weights * r) / w_mean
        if qe is not None:
            t = fr & (qe > 0.05) & (qe < 0.95)
            row['rms_frames'] = np.sqrt(np.sum(d.path_weights[t] * (r[t] - qe[t]) ** 2) / np.sum(d.path_weights[t]))
            pe = bm(d.path_weights * qe) / w_mean
            tb = (pe > 0.05) & (pe < 0.95)
            row['rms_bins_lambda'] = np.sqrt(np.nanmean((profiles[(name, s)] - pe)[tb] ** 2))
            profiles[('exact', name)] = pe
        rows.append(row)
summary = pd.DataFrame(rows).set_index(['weighting', 'seed'])
os.makedirs(OUT, exist_ok=True)
summary.to_csv(f'{OUT}/summary.csv')
display(summary.round(4))

edges = edges_for(data)
mid = (edges[0][1:] + edges[0][:-1]) / 2
has_exact = any(k[0] == 'exact' for k in profiles)
fig, axes = plt.subplots(2 if has_exact else 1, 1, figsize=(9, 7 if has_exact else 4), sharex=True, squeeze=False)
for (name, s), P in profiles.items():
    if name == 'exact':
        continue
    axes[0, 0].plot(mid, P, label=f'{name}, seed {s}')
    if has_exact:
        axes[1, 0].plot(mid, P - profiles[('exact', name)], label=f'{name}, seed {s}')
if has_exact:
    axes[0, 0].plot(mid, profiles[('exact', next(iter(fits)))], 'k--', label='exact')
    axes[1, 0].axhline(0, color='k', lw=0.8)
    axes[1, 0].set(ylabel='fit $-$ exact')
    axes[1, 0].grid(alpha=0.3)
axes[0, 0].set(ylabel='$q$ (mean per bin)')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[-1, 0].set(xlabel=XLABEL)
fig.tight_layout()
fig.savefig(f'{OUT}/profiles.png', dpi=110)
plt.show()

**How q depends on every CV.** For every fit, the path-weighted mean q per bin of each CV (over the frames outside A and B) and the share of the variance of q that this CV alone explains, η² = Var(E[q | CV bin]) / Var(q). A CV with a high η² is closely related to the committor; a CV with a flat mean q does not matter by itself (it can still matter together with the others). The mean q curves are the committor averaged over all the other CVs at fixed value of this one, so they show where the CV acts, not a fit as a function of it alone.

In [ ]:
name0 = next(iter(fits))
d = datas[name0]
r0 = next(iter(fits[name0]['r'].values()))
fr = ~(d.boundary0 | d.boundary1)
w = d.path_weights
names = getattr(d, 'cv_names', [f'cv{j}' for j in range(d.X.shape[1])])
n_cv = d.X.shape[1]
ncol = 4
nrow = int(np.ceil(n_cv / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3 * nrow), squeeze=False, sharey=True)
q_mean = np.sum(w[fr] * r0[fr]) / np.sum(w[fr])
q_var = np.sum(w[fr] * (r0[fr] - q_mean) ** 2) / np.sum(w[fr])
eta = {}
for j, (ax, n) in enumerate(zip(axes.ravel(), names)):
    x = d.X[fr, j]; ww = w[fr]; qq = r0[fr]
    edges_j = np.quantile(x, np.linspace(0, 1, 41))                       # bins of equal frame count
    edges_j = np.unique(edges_j)
    bi = np.clip(np.digitize(x, edges_j) - 1, 0, len(edges_j) - 2)
    wb = np.bincount(bi, ww, len(edges_j) - 1)
    qb = np.bincount(bi, ww * qq, len(edges_j) - 1) / np.maximum(wb, 1e-300)
    eta[n] = np.sum(wb * (qb - q_mean) ** 2) / np.sum(wb) / q_var
    ax.plot((edges_j[1:] + edges_j[:-1]) / 2, qb)
    ax.set(xlabel=CV_LABELS.get(n, n) if 'CV_LABELS' in globals() else n, title=f'{n}: $\\eta^2$ = {eta[n]:.2f}')
    ax.grid(alpha=0.3)
for ax in axes.ravel()[n_cv:]:
    ax.axis('off')
for ax in axes[:, 0]:
    ax.set_ylabel('mean $q$')
fig.tight_layout()
fig.savefig(f'{OUT}/q_vs_cvs.png', dpi=110)
plt.show()
display(pd.Series(eta, name='eta2').sort_values(ascending=False).round(3).to_frame())


## 8. Save

Every fit's q per frame with its frames, in the format of `load_tis_data_cupy.py`'s `committor.npz` (`r`, `lam`, `X_all`, `i_traj`, `t_traj`, `group_traj`, `path_weights`, `boundary0`, `boundary1`), as `<OUT>/<weighting>/seed<s>/committor.npz`. (`run_seeds` also saved each final q alone as `q_traj.npy`.)

In [ ]:
for name, fit in fits.items():
    d = datas[name]
    for s, r in fit['r'].items():
        path = f'{OUT}/{name}/seed{s}/committor.npz'
        np.savez(path, r=r, lam=d.lam, X_all=d.X_all, i_traj=d.i_traj, t_traj=d.t_traj, group_traj=d.group_traj,
                 path_weights=d.path_weights, boundary0=d.boundary0, boundary1=d.boundary1,
                 weighting=name, tis_dir=TIS_DIR)
        print('saved', path)